In [2]:
import os
import torch
import pandas as pd
import geopandas as gpd

import numpy as np
np.seterr(divide='ignore', invalid='ignore', over='ignore')

import paths
from tqdm import tqdm

from train_model import load_data, train
from train_model import RE_utils as ut
from apollo import mechanics as ma

In [3]:
import importlib

In [4]:
days = 6

years_train = [1980 + i for i in range(30)]
years_eval = [2010 + i for i in range(10)]

features = (['Rain'] + ['Rain-' + f'{d+1}' for d in range(days)] 
            + ['Temperature'] + ['Temperature-' + f'{d+1}' for d in range(days)] \
            + ['Resultant Windspeed'] + ['Resultant Windspeed-' + f'{d+1}' for d in range(days)] \
            + ['Humidity'] + ['Humidity-' + f'{d+1}' for d in range(days)]\
            #+ ['Soil Moisture ' + f'{i+1}' for i in range(4)])
            + ['Rain_28_Mu','Rain_90_Mu','Rain_180_Mu', 'Temperature_28_Mu','Temperature_90_Mu','Temperature_180_Mu']) 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
def train_NN(rf, years_train, years_eval, features=features, loss_func=None, alpha=1.0, beta=1.0, grid_search=False, early_stopping=True, verbose=True, network_params={}):

    # LOAD DATA
    trnset, full_set = load_data.preprocess_data(rf, features, years_eval, years_train)
    targets = ['Flow']
    xspace = ma.featurelocator(rf, features)
    yspace = ma.featurelocator(rf, targets)

    x_train = load_data.reshape_input(trnset, xspace)
    y_train = load_data.reshape_output(trnset, yspace)   
    
    psi = ut.psi_distribution(y_train, 'lognorm', alpha=alpha, beta=beta, plot=False)
    
    # TRAINING
    network = train.train(x_train, 
                          y_train,
                          verbose=verbose, 
                          loss_func_type=loss_func, 
                          psi=psi[:len(x_train)], 
                          grid_search=grid_search, 
                          early_stopping=early_stopping,
                          network_params=network_params)

    # EVALUATION
    x_eval = load_data.reshape_input(full_set, xspace)  
    y_eval = load_data.reshape_input(full_set, yspace)  
    
    rf.loc[:, 'Predicted'] = network.predict(torch.from_numpy(x_eval).to(device))
    rf.loc[:, 'Groundtruth'] = y_eval    
    rf['Date'] =  pd.to_datetime(rf['Date'], unit='s')
    rf = rf[rf['Date'].dt.year.isin(years_train + years_eval)]
    rf.reset_index(drop=True, inplace=True)
    return rf, network

In [6]:
importlib.reload(train)

<module 'train_model.train' from '/Users/av656/Documents/Cambridge/PhD Research/Coding/Lowlands-vs_highlands-streamflow/train_model/train.py'>

In [8]:
## MODEL TRAINING + PREDICTIONS

input_type = '_9to9'
days = 6

for station_nr in tqdm(os.listdir(paths.CATCHMENT_BASINS), desc="Processing Stations"):
    
    out_path = paths.PREDICTIONS + f"/era5_run4/{station_nr}_era5_run4.csv"
    #out_path = paths.PREDICTIONS + f"/nrfa_run4/{station_nr}_nrfa_run4.csv"
    
    if not os.path.exists(out_path):
        catchment_boundary = gpd.read_file(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + '.shp')
    
        original_data = pd.read_csv(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + f"_lumped{input_type}.csv", low_memory=False)
        original_data['Date'] = pd.to_datetime(original_data['Date'], format='%Y-%m-%d').dt.date
        #subset = ['Flow'] + ['Rain'] + ['Rain-' + f'{d+1}' for d in range(days)] + ['Rain_28_Mu','Rain_90_Mu','Rain_180_Mu']
        #clean_data = original_data #.dropna(subset=subset)
        expanded_features = features + ['Snow'] + ['Snow-' + f'{d+1}' for d in range(days)]
    
        outdf, network = train_NN(rf=original_data, years_train=years_train, years_eval=years_eval, features=expanded_features, loss_func=None, alpha=1, beta=2, grid_search=False, verbose=False)
        outdf.to_csv(out_path)

Processing Stations: 100%|██████████| 899/899 [3:44:05<00:00, 14.96s/it]  


In [11]:
os.path.exists(paths.DATA)

True

In [10]:
## MODEL TRAINING + PREDICTIONS

input_type = '_9to9'
days = 6

test_station = os.listdir(paths.CATCHMENT_BASINS)[0]
    
catchment_boundary = gpd.read_file(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + '.shp')
    
original_data = pd.read_csv(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + f"_lumped{input_type}.csv", low_memory=False)

original_data['Date'] = pd.to_datetime(original_data['Date'], format='%Y-%m-%d').dt.date
#subset = ['Flow'] + ['Rain'] + ['Rain-' + f'{d+1}' for d in range(days)] + ['Rain_28_Mu','Rain_90_Mu','Rain_180_Mu']
#clean_data = original_data #.dropna(subset=subset)
expanded_features = features + ['Snow'] + ['Snow-' + f'{d+1}' for d in range(days)]
    
#outdf, network = train_NN(rf=original_data, years_train=years_train, years_eval=years_eval, features=expanded_features, loss_func=None, alpha=1, beta=2, grid_search=False, verbose=False)
#outdf.to_csv(out_path)

original_data

,Unnamed: 0,Date,Flow,pressure_level,Rain,Snow,Temperature,U Windspeed,V Windspeed,Humidity,...,Humidity-21,Humidity-22,Humidity-23,Humidity-24,Humidity-25,Humidity-26,Humidity-27,Humidity_28_Mu,Humidity_90_Mu,Humidity_180_Mu
0,179,1979-06-29,1.2,1000.0,3.276140,0.000000e+00,283.840193,8.262467,2.089924,75.757016,...,81.576980,68.591614,76.248308,76.667017,79.463765,71.454366,50.805341,74.565023,72.029144,75.780893
1,180,1979-06-30,0.0,1000.0,0.054523,0.000000e+00,283.211735,5.921340,-3.546461,62.382135,...,62.199868,81.576980,68.591614,76.248308,76.667017,79.463765,71.454366,74.978480,71.844756,75.695284
2,181,1979-07-01,0.0,1000.0,0.093609,0.000000e+00,285.141432,6.421828,-1.766367,66.436943,...,69.899786,62.199868,81.576980,68.591614,76.248308,76.667017,79.463765,74.799287,71.695109,75.548055
3,182,1979-07-02,0.0,1000.0,0.007299,0.000000e+00,285.534195,5.471475,0.408015,76.837511,...,73.703649,69.899786,62.199868,81.576980,68.591614,76.248308,76.667017,74.705492,71.903364,75.545200
4,183,1979-07-03,0.0,1000.0,0.001403,0.000000e+00,286.729252,1.817952,1.288850,66.527965,...,78.338531,73.703649,69.899786,62.199868,81.576980,68.591614,76.248308,74.343383,71.804317,75.525489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14061,14240,2017-12-27,0.0,1000.0,0.000297,0.000000e+00,274.584035,2.013772,-1.829821,81.597173,...,80.391258,89.841609,89.691202,87.533819,94.607551,73.108034,68.669582,85.998478,82.701972,78.919856
14062,14241,2017-12-28,4.6,1000.0,1.933198,5.676698e-08,277.053193,3.340716,-0.928061,74.244592,...,75.456531,80.391258,89.841609,89.691202,87.533819,94.607551,73.108034,86.197585,82.739765,78.931558
14063,14242,2017-12-29,21.5,1000.0,7.955138,6.609736e-05,276.864178,-1.154462,3.254249,92.817317,...,65.200903,75.456531,80.391258,89.841609,89.691202,87.533819,94.607551,86.901488,82.902157,79.044248
14064,14243,2017-12-30,17.2,1000.0,8.506813,2.950642e-05,279.953211,2.605201,3.595040,88.926667,...,80.049623,65.200903,75.456531,80.391258,89.841609,89.691202,87.533819,86.698600,82.878293,79.215564


In [ ]:
for station_nr in tqdm(os.listdir(paths.CATCHMENT_BASINS), desc="Processing Stations"):
    
    out_path = paths.PREDICTIONS + f"/era5_run4/{station_nr}_era5_run4.csv"
    #out_path = paths.PREDICTIONS + f"/nrfa_run4/{station_nr}_nrfa_run4.csv"
    
    if not os.path.exists(out_path):
        catchment_boundary = gpd.read_file(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + '.shp')
    
        original_data = pd.read_csv(paths.CATCHMENT_BASINS + '/' + str(station_nr) + '/' + str(station_nr) + f"_lumped{input_type}.csv", low_memory=False)
        original_data['Date'] = pd.to_datetime(original_data['Date'], format='%Y-%m-%d').dt.date
        #subset = ['Flow'] + ['Rain'] + ['Rain-' + f'{d+1}' for d in range(days)] + ['Rain_28_Mu','Rain_90_Mu','Rain_180_Mu']
        #clean_data = original_data #.dropna(subset=subset)
        expanded_features = features + ['Snow'] + ['Snow-' + f'{d+1}' for d in range(days)]